In [2]:
import numpy as np

In [3]:
class MLP:
    def __init__(self, n_hidden, n_output=1, lr=0.1, activation="sigmoid", epochs=10):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = 32
        self.hidden_nodes = n_hidden
        self.output_nodes = n_output
        
        self.bias_h = np.random.uniform(-1, 1, (self.hidden_nodes, 1))
        self.bias_o = np.random.uniform(-1, 1, (self.output_nodes, 1))
        self.weights_ih = None
        self.weights_ho = None

        self.activation_name = activation
        if activation == "sigmoid":
            self.activation = self.sigmoid
        elif activation == "relu":
            self.activation = self.relu
        elif activation == "arctan":
            self.activation = self.arctan
        elif activation == "cosine":
            self.activation = self.cosine
        
    def sigmoid(self, x):  return 1/(1 + np.exp(-x))
    def relu(self, x):     return np.maximum(0, x)
    def arctan(self, x):   return np.arctan(x)
    def cosine(self, x):   return np.cos(x)

    def getderivatives(self, hidden_sum):
        if self.activation_name == "sigmoid":
            hidden_out = self.sigmoid(hidden_sum)
            return hidden_out * (1 - hidden_out)
            
        elif self.activation_name == "relu":
            return (hidden_sum > 0).astype(float)
            
        elif self.activation_name == "arctan":
            return 1 / (1 + np.power(hidden_sum, 2))
            
        elif self.activation_name == "cosine":
            return -np.sin(hidden_sum)

    
    def feedforward(self, inputs):
        inputs = np.array(inputs)
        self.hidden_sum = np.matmul(self.weights_ih, np.transpose(inputs)) + self.bias_h
        self.hidden_out = self.activation(self.hidden_sum)
        self.output_sum = np.matmul(self.weights_ho, self.hidden_out) + self.bias_o
        self.output_out = self.sigmoid(self.output_sum)
        return self.output_out.reshape(-1, 1)

    
    def backprop(self, X, y):
        y = np.array(y)
        X = np.array(X)
        self.feedforward(X)
        output_deltas = y - self.output_out
        hidden_errors = np.matmul(self.weights_ho.T, output_deltas)
        hidden_deltas = hidden_errors * self.getderivatives(self.hidden_sum)

        batch_size = X.shape[0]
        self.weights_ho += self.lr * np.matmul(output_deltas, self.hidden_out.T) / batch_size
        self.weights_ih += self.lr * np.matmul(hidden_deltas, X) / batch_size
        self.bias_h += self.lr * np.sum(hidden_deltas, axis=1, keepdims=True) / batch_size
        self.bias_o += self.lr * np.sum(output_deltas, axis=1, keepdims=True) / batch_size


    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        if self.weights_ih is None:
            self.input_nodes = X.shape[1]
            limit = np.sqrt(6 / (self.input_nodes + self.output_nodes))
            self.weights_ih = np.random.uniform(-limit, limit, (self.hidden_nodes, self.input_nodes))
            self.weights_ho = np.random.uniform(-limit, limit, (self.output_nodes, self.hidden_nodes))

        for epoch in range(self.epochs):
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)
            X = X[indices]
            y = y[indices]

            for i in range(0, X.shape[0], self.batch_size):
                X_batch = X[i:i+self.batch_size]
                y_batch = y[i:i+self.batch_size]
                self.backprop(X_batch, y_batch)


    def predict_proba(self, X):
        if self.weights_ih is None:
            print("Train the model first.")
            return
        X = np.array(X)
        output = self.feedforward(X)
        prob_0 = 1 - output    
        return np.hstack([prob_0, output])

    def predict(self, X):
        probas = self.predict_proba(X)[:, 1]
        return (probas >= 0.5).astype(int)